https://api.secondarymetabolites.org/#antismash-suite-api

what you want to do is to use seq to provide the file instead of ncbi in the example
and maybe genefinding=none

--genefinding-tool none --cb-knownclusters --cb-subclusters --cc-mibig --clusterhmmer --pfam2go --rre --asf --no-abort-on-invalid-records --cb-general --tigrfam

all of the -- parameters should match to form fields of the same name, I think


oh, you need to replace the inner dashes with underscores as well

and the clusterblast options are clusterblast, knownclusterblast, and subclusterblast, the way those parameters were called a decade ago

In [27]:
# KnownClusterBlast
# --cb-knownclusters
# "knownclusterblast": True

# SubClusterBlast
# --cb-subclusters
# "subclusterblast": True

# ClusterCompare
# --cc-mibig
# "clusterblast": True

# --genefinding-tool none 
# "genefinder": None

# --clusterhmmer

# --pfam2go 

# --rre 

# --asf 

# --no-abort-on-invalid-records 

# --cb-general 

# --tigrfam

SyntaxError: invalid syntax (2799958329.py, line 1)

In [2]:
import logging
import os
import shutil
import time
from os import PathLike
from pathlib import Path
from typing import Optional
from typing import Union
import requests
from nplinker.utils import check_md5
from nplinker.utils import download_and_extract_archive
from nplinker.utils import download_url
from nplinker.utils import extract_archive
from nplinker.utils import list_dirs
from nplinker.utils import list_files


In [3]:
download_root =  "/Users/alien/coding/NPLinker_workshop_2025/antismash_api/downloads"
extract_root =  "/Users/alien/coding/NPLinker_workshop_2025/antismash_api"

# Step 1: Download the genomes from NCBI

In [4]:
def download_and_extract_ncbi_genome(
    refseq_id: str, download_root: Union[str, PathLike], max_retries: int = 10
) -> Optional[Path]:
    """Downloads and extracts an NCBI dataset for a given genome refseq ID.

    This function attempts to download a dataset from the NCBI database using
    the provided refseq ID. It retries the download and extraction process up
    to a maximum number of times if any errors occur. The function verifies
    the integrity of the downloaded files using MD5 checksums and moves and
    renames the GenBank files upon successful verification.

    Args:
        refseq_id (str): The refseq ID for the dataset to be downloaded.
        download_root (str or Path): The root directory where the dataset will be downloaded.
        max_retries (int): The maximum number of times to retry downloading and extracting

    Returns:
        Path: The path to the extracted dataset if successful, otherwise None.

    Raises:
        Exception: If the maximum number of retries is reached and the dataset could
        not be successfully downloaded and extracted.
    """
    url = (
        "https://api.ncbi.nlm.nih.gov/datasets/v2/genome/accession/"
        f"{refseq_id}/download?include_annotation_type=GENOME_GB"
    )

    download_root = Path(download_root)
    extract_path = download_root / "ncbi_genomes"
    filename = f"ncbi_{refseq_id}.zip"

    extract_path.mkdir(parents=True, exist_ok=True)

    for attempt in range(1, max_retries + 1):
        try:
            logging.info(
                f"Attempt {attempt}/{max_retries}: Downloading and extracting ncbi genome {refseq_id}..."
            )

            download_url(url, download_root, filename)
            archive = download_root / filename

            extract_archive(archive, extract_path)

            md5_ok = verify_ncbi_dataset_md5_sums(extract_path)
            if md5_ok:
                logging.info("MD5 checksums verified. Moving and renaming GenBank file.")

                # Move and rename GenBank file
                genbank_path = extract_path / "ncbi_dataset" / "data" / refseq_id / "genomic.gbff"
                new_genbank_path = extract_path / f"{refseq_id}.gbff"
                genbank_path.rename(new_genbank_path)

                # Delete unnecessary files
                shutil.rmtree(extract_path / "ncbi_dataset")
                os.remove(extract_path / "md5sum.txt")
                os.remove(extract_path / "README.md")

                return new_genbank_path
            else:
                logging.error("MD5 checksums verification failed.")

        except Exception as e:
            logging.error(f"Error occurred during attempt {attempt}: {e}")

    logging.error("Maximum retries reached. Download and extraction failed.")
    return None


def verify_ncbi_dataset_md5_sums(extract_path: Path) -> bool:
    """Check MD5 checksums for files in the extraction path.

    Args:
        extract_path (Path): Path to the extraction directory.

    Returns:
        bool: True if all MD5 checksums are correct, False otherwise.
    """
    md5_ok = True
    with open(extract_path / "md5sum.txt", "r") as f:
        for line in f:
            md5sum, file_name = line.strip().split()
            file_path = extract_path / file_name
            if check_md5(file_path, md5sum):
                logging.info(f"MD5 checksum for {file_name} is correct")
            else:
                logging.error(f"MD5 checksum for {file_name} is incorrect")
                md5_ok = False
    return md5_ok

In [5]:
refseq_id = "GCF_009711935.1"

genbank_file_path = download_and_extract_ncbi_genome(refseq_id, download_root)
genbank_file_path

Output()

PosixPath('/Users/alien/coding/NPLinker_workshop_2025/antismash_api/downloads/ncbi_genomes/GCF_009711935.1.gbff')

# Step 2: Submit antimash jobs using API

In [78]:
job_id = initiate_antismash_job(genbank_file_path)
print(job_id)

bacteria-75434feb-d663-4a48-bf15-cea488853cfb


In [79]:
import time 

while True:
    response = query_antismash_job_status(job_id)
    job_status = response.get("status") if response else None
    print(job_status)
    if job_status == "done":
        break
    time.sleep(15)

running:  Comparing regions to reference database
running:  HMM detection using strictness: relaxed
running:  HMM detection using strictness: relaxed
running:  Comparing regions to reference database
running:  Comparing regions to reference database
running:  Running antismash.modules.sactipeptides
running:  Running antismash.modules.lanthipeptides
running:  HMM detection using strictness: relaxed
running:  Comparing regions to reference database
running:  Comparing regions to reference database
running:  HMM detection using strictness: relaxed
running:  Comparing regions to reference database
running:  Running antismash.modules.lanthipeptides
running:  Running antismash.modules.lassopeptides
running:  Running antismash.modules.lanthipeptides
running:  Comparing regions to reference database
running:  Running antismash.modules.lassopeptides
running:  Comparing regions to reference database
running:  Running antismash.modules.lanthipeptides
running:  Comparing regions to reference datab

In [80]:
download_and_extract_antismash_api_results(
    job_id, refseq_id, download_root, extract_root
)

Output()

In [77]:
def initiate_antismash_job(genbank_filepath: PathLike) -> Optional[str]:
    """Submits an antiSMASH job using the provided GenBank file.

    This function submits a job to the antiSMASH API and
    returns the job ID if the submission is successful. If an HTTP error or any
    other exception occurs during the submission, it prints an error message and
    returns None.

    Args:
        genbank_filepath (PathLike): The path to the GenBank file to be submitted to the antiSMASH API.

    Returns:
        Optional[str]: The job ID if successful, otherwise None.
    """
    url = 'https://antismash.secondarymetabolites.org/api/v1.0/submit'

    try:
        with open(genbank_filepath, 'rb') as file:
            files = {'seq': file}
            data = {
                'knownclusterblast': 'true',
                'cc_mibig': 'true',
            }
            response = requests.post(url, files=files, data=data)
            response.raise_for_status()  # Raise an exception for HTTP errors

            job_id = response.json().get('id')
            if job_id:
                logging.info(
                    f"Successfully submitted job for file '{genbank_filepath.name}' "
                    f"with Job ID: {job_id}"
                )
                return job_id
            else:
                logging.error(f"Failed to submit job for file {genbank_filepath.name}")
                return None

    except requests.exceptions.RequestException as req_err:
        logging.error(f"Request failed: {req_err}")
    except ValueError as json_err:  # Handles JSON decoding errors
        logging.error(f"Invalid JSON response: {json_err}")
    except Exception as err:
        logging.error(f"Unexpected error: {err}")

    return None

In [42]:
job_id = initiate_antismash_job(genbank_file_path)
job_id

'bacteria-6b7a1e5d-2e6b-468c-824c-d5df1d37bccd'

# Step 3: Wait for the job to complete

In [8]:
def query_antismash_job_status(job_id: str) -> Optional[dict]:
    """Gets the status of an antiSMASH job.

    Args:
        job_id (str): The job ID to query.

    Returns:
        dict: The response JSON if successful, otherwise None.
    """
    url = f"https://antismash.secondarymetabolites.org/api/v1.0/status/{job_id}"

    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()  # Raise an exception for HTTP errors
        return response.json()

    except requests.exceptions.RequestException as req_err:
        logging.error(f"Request failed for job_id {job_id}: {req_err}")
    except ValueError as json_err:  # Handles JSON decoding errors
        logging.error(f"Invalid JSON response for job_id {job_id}: {json_err}")
    except Exception as err:
        logging.error(f"Unexpected error while getting job state for job_id {job_id}: {err}")

    return None


def wait_for_antismash_job_completion(
    job_id: str, max_wait_time: int = 1200, poll_interval: int = 20
) -> Optional[dict]:
    """Waits for the antiSMASH job to complete by polling the job status.

    Args:
        job_id (str): The job ID to query.
        max_wait_time (int): Maximum wait time in seconds. Default is 1200 seconds (= 20 min).
        poll_interval (int): Time interval between successive polls in seconds. Default is 20 seconds.

    Returns:
        dict: The full response JSON if the job completes successfully, otherwise None.
    """
    start_time = time.time()

    while (time.time() - start_time) < max_wait_time:
        response = query_antismash_job_status(job_id)

        if response is None:
            logging.error(f"Failed to retrieve job status for job_id {job_id}. Exiting wait loop.")
            return None

        job_state = response.get("state")
        if not job_state:
            logging.error(f"Job state missing in response for job_id: {job_id}")
            return response

        if job_state == "failed":
            job_status = response.get("status", "No error message provided")
            logging.error(f"AntiSMASH job {job_id} failed with an error: {job_status}")
            return response

        if job_state == "done":
            logging.info(f"Job {job_id} completed successfully.")
            return response

        # Wait before polling again
        time.sleep(poll_interval)

    logging.error(f"Job {job_id} did not complete within {max_wait_time} seconds.")
    return None


In [12]:
response = wait_for_antismash_job_completion(job_id)
job_status = response.get('status') if response else None
job_status

'done'

# Step 4: Download the antismash results

In [22]:
ANTISMASH_API_DOWNLOAD_URL = "https://antismash.secondarymetabolites.org/upload/{}/{}"

def download_and_extract_antismash_api_results(
    job_id: str, refseq_id: str, download_root: Union[str, PathLike], extract_root: Union[str, PathLike]
) -> None:
    """Downloads and extracts results from an antiSMASH API job for a given job ID and refseq ID.

    This function constructs the download URL using the provided job ID and refseq ID, then
    downloads the results as a ZIP file and extracts its contents to the specified directories.

        download_root (str or PathLike): The root directory where the ZIP file will be downloaded.
        extract_root (str or PathLike): The root directory where the contents of the ZIP file will be extracted.

    Raises:
        requests.exceptions.RequestException: If there is an issue with the HTTP request.
        zipfile.BadZipFile: If the downloaded file is not a valid ZIP file.
        OSError: If there is an issue with file operations such as writing or extracting.
    """
    url = ANTISMASH_API_DOWNLOAD_URL.format(job_id, refseq_id + ".zip")
    _download_and_extract_antismash(url, refseq_id, download_root, extract_root)


def _download_and_extract_antismash(
    url: str, refseq_id: str, download_root: Union[str, PathLike], extract_root: Union[str, PathLike]
) -> bool:
    download_root = Path(download_root)
    extract_root = Path(extract_root)
    extract_path = extract_root / "antismash" / refseq_id

    try:
        if extract_path.exists():
            _check_extract_path(extract_path)
        else:
            extract_path.mkdir(parents=True, exist_ok=True)

        download_and_extract_archive(url, download_root, extract_path, refseq_id + ".zip")
        #_cleanup_extracted_files(extract_path)

        logging.info(f"antiSMASH BGC data of {refseq_id} is downloaded and extracted.")

    except Exception as e:
        shutil.rmtree(extract_path)
        logging.warning(e)
        raise e


def _check_extract_path(extract_path: Path):
    # check if extract_path is empty
    if any(extract_path.iterdir()):
        raise ValueError(f'Nonempty directory: "{extract_path}"')


def _cleanup_extracted_files(extract_path: Path) -> None:
    # delete subdirs
    for subdir_path in list_dirs(extract_path):
        shutil.rmtree(subdir_path)

    # delete unnecessary files
    files_to_keep = list_files(extract_path, suffix=(".json", ".gbk"))
    for file in list_files(extract_path):
        if file not in files_to_keep:
            os.remove(file)


In [19]:
download_and_extract_antismash_api_results(job_id, refseq_id, download_root, extract_root)

Output()

In [29]:
def main():
    download_root = "/Users/alien/coding/NPLinker_workshop_2025/antismash_api/downloads"
    extract_root = "/Users/alien/coding/NPLinker_workshop_2025/antismash_api"

    refseq_ids = ["GCF_009711985.1", "GCF_000204075.1"]
    job_ids = []

    for refseq_id in refseq_ids:
        genbank_file_path = download_and_extract_ncbi_genome(refseq_id, download_root)
        job_id = initiate_antismash_job(genbank_file_path)
        job_ids.append(job_id)
        print(refseq_id, job_id)

    for refseq_id, job_id in zip(refseq_ids, job_ids):
        response = wait_for_antismash_job_completion(job_id)
        job_status = response.get("status") if response else None
        if job_status == "done":
            download_and_extract_antismash_api_results(
                job_id, refseq_id, download_root, extract_root
            )

In [30]:
main()

ERROR:root:Error occurred during attempt 1: Failed to download url https://api.ncbi.nlm.nih.gov/datasets/v2/genome/accession/GCF_009711985.1/download?include_annotation_type=GENOME_GB with status code 429
ERROR:root:Error occurred during attempt 2: Failed to download url https://api.ncbi.nlm.nih.gov/datasets/v2/genome/accession/GCF_009711985.1/download?include_annotation_type=GENOME_GB with status code 429
ERROR:root:Error occurred during attempt 3: Failed to download url https://api.ncbi.nlm.nih.gov/datasets/v2/genome/accession/GCF_009711985.1/download?include_annotation_type=GENOME_GB with status code 429
ERROR:root:Error occurred during attempt 4: Failed to download url https://api.ncbi.nlm.nih.gov/datasets/v2/genome/accession/GCF_009711985.1/download?include_annotation_type=GENOME_GB with status code 429
ERROR:root:Error occurred during attempt 5: Failed to download url https://api.ncbi.nlm.nih.gov/datasets/v2/genome/accession/GCF_009711985.1/download?include_annotation_type=GENOME

Output()

GCF_009711985.1 bacteria-442991d1-fca1-4bac-9f0f-adf898ebc1bf


ERROR:root:Error occurred during attempt 1: Failed to download url https://api.ncbi.nlm.nih.gov/datasets/v2/genome/accession/GCF_000204075.1/download?include_annotation_type=GENOME_GB with status code 429


Output()

GCF_000204075.1 bacteria-5d415ea4-3a4b-4f1d-b377-6cf52dcf0e90


Output()

Output()